# Neural Identifier Training with Particle Filters - Nonlinear Pendulum

In [9]:
import numpy as np
import plotly.graph_objects as go

In [10]:
# ============================================================
# 1) True nonlinear system (Nonlinear Pendulum)
# ============================================================
def plant_dynamics(x, u, L=1.0, m=1.0, g=9.81, b=0.1):
    """
    Continuous dynamics for nonlinear pendulum: x = [theta, theta_dot].
    Returns x_dot.
    
    The nonlinear pendulum equations:
    dtheta/dt = theta_dot
    dtheta_dot/dt = -(g/L) * sin(theta) - (b/(m*L^2)) * theta_dot + u/(m*L^2)
    
    where:
    theta: angular position (rad)
    theta_dot: angular velocity (rad/s)
    u: applied torque (N·m)
    L: pendulum length (m)
    m: pendulum mass (kg)
    g: gravitational acceleration (m/s^2)
    b: damping coefficient (N·m·s/rad)
    """
    theta, theta_dot = x
    u_torque = u[0] if isinstance(u, (list, np.ndarray)) else u
    
    # Nonlinear pendulum dynamics
    theta_ddot = -(g/L) * np.sin(theta) - (b/(m*L**2)) * theta_dot + u_torque/(m*L**2)
    
    return np.array([theta_dot, theta_ddot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          friction_variation=0.02, sensor_bias=[0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic pendulum disturbances.
    
    Args:
        x_k: current state [theta, theta_dot]
        u_k: control input [torque] 
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        friction_variation: friction coefficient variations
        sensor_bias: systematic biases in measurements
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic pendulum disturbances
    
    # 1. Friction variations (velocity-dependent)
    friction_noise = friction_variation * np.array([
        0.0,  # No direct effect on theta
        np.sign(x_kp1[1]) * np.abs(x_kp1[1]) * np.random.randn()  # Friction affects theta_dot
    ])
    
    # 2. Control-dependent noise (increases with torque magnitude)
    control_magnitude = np.abs(u_k[0]) if isinstance(u_k, (list, np.ndarray)) else np.abs(u_k)
    control_noise_factor = 1 + 0.1 * control_magnitude
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * control_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 3, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * control_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * control_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * control_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Encoder quantization effects
    encoder_resolution = 0.001  # 0.001 rad resolution
    quantization_noise = encoder_resolution * (np.random.rand(2) - 0.5)
    
    # Combine all disturbances
    x_kp1 += friction_noise + total_noise + bias_noise + quantization_noise
    
    # 6. Angle wrapping for realistic behavior
    x_kp1[0] = np.arctan2(np.sin(x_kp1[0]), np.cos(x_kp1[0]))  # wrap angle to [-π, π]
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='sine'):
    """
    Generate realistic control inputs for pendulum.
    
    Args:
        t: time value
        trajectory_type: 'sine', 'square', 'step', 'mixed', 'swing_up'
    
    Returns:
        u: [torque] control torque
    """
    if trajectory_type == 'sine':
        # Sinusoidal torque
        torque = 2.0 * np.sin(0.5 * t)
        return np.array([torque])
    
    elif trajectory_type == 'square':
        # Square wave torque
        period = 8.0  # 8 second period
        torque = 1.5 if (t % period) < (period / 2) else -1.5
        return np.array([torque])
    
    elif trajectory_type == 'step':
        # Step inputs
        if t < 5.0:
            torque = 1.0
        elif t < 10.0:
            torque = -1.0
        elif t < 15.0:
            torque = 0.5
        else:
            torque = 0.0
        return np.array([torque])
    
    elif trajectory_type == 'swing_up':
        # Energy-based swing-up control
        k = 0.5  # Control gain
        target_energy = 20.0  # Target energy for upright position
        # Simple energy-based control (requires state feedback - simplified here)
        torque = k * np.sin(2 * t) * np.exp(-0.1 * t)
        return np.array([torque])
    
    else:  # 'mixed' - combination of different behaviors
        # Mixed trajectory with different phases
        phase = (t % 20.0) / 20.0  # 20-second cycles
        
        if phase < 0.25:  # Sine wave
            torque = 1.5 * np.sin(3 * t)
        elif phase < 0.5:  # Step input
            torque = 1.0
        elif phase < 0.75:  # Negative sine
            torque = -1.0 * np.sin(2 * t)
        else:  # Damped oscillation
            torque = 0.5 * np.sin(5 * t) * np.exp(-0.1 * (t % 5))
        
        return np.array([torque])

In [11]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input=None):
    """
    Features for a 2-state pendulum system with control input:
    x = [theta, theta_dot], u = [torque]
    
    z = [S(θ), S(θ_dot), S(θ)S(θ_dot), S(θ)^2, S(θ_dot)^2, 
         cos(θ), sin(θ), θ, θ_dot, S(u), u, 1]
    """
    s_theta = sigmoidal(x_est[0])      # angular position
    s_theta_dot = sigmoidal(x_est[1])  # angular velocity
    
    # Basic features
    features = [
        s_theta, s_theta_dot,                 # Individual sigmoid terms
        s_theta * s_theta_dot,                # Cross term
        s_theta**2, s_theta_dot**2,           # Quadratic terms
        np.cos(x_est[0]), np.sin(x_est[0]),   # Trigonometric terms (important for pendulum)
        x_est[0], x_est[1],                   # Direct state terms
    ]
    
    # Add control input features if available
    if u_input is not None and len(u_input) >= 1:
        s_u = sigmoidal(u_input[0])           # torque sigmoid term
        features.extend([
            s_u,                              # Control sigmoid term
            u_input[0],                       # Direct control term
        ])
    else:
        # Add zero placeholders if no control input
        features.extend([0.0, 0.0])
    
    # Add bias term
    features.append(1.0)                      # Bias term
    
    return np.array(features)


def RHONN_predict(x_state_for_z, w_neuron, u_input=None):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_input)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [12]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [13]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : filter's own estimate at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        u_input: control input at time k (for mobile robot)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)
        z = construct_z_vector(x_state_for_z, u_input)  # (num_features,)

        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

        # 2) Update: importance weights with Gaussian likelihood using state-specific R_var
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability using state-specific R_var
            ll = -0.5 * (innov**2) / self.R_var[i]
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return information about the PF parameters for each state."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i],
                'R_var': self.R_var[i]
            }
        return info

In [14]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [16]:
# ============================================================
# 5) Simulation Main Loop
# ============================================================

# --- Simulation settings ---
n_steps = 1500
dt = 0.02
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'mixed'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.01
friction_variation = 0.01
sensor_bias = [0.001, 0.0005]  # Small systematic biases [theta, theta_dot]

# --- True system init ---
x_true = np.zeros((n_steps, 2))
x_true[0] = [0.5, 0.0]  # Initial conditions for pendulum [theta, theta_dot]

# --- Control trajectory ---
trajectory_type = 'sine'  # 'sine', 'square', 'step', 'mixed', 'swing_up'

# --- RHONN config ---
num_neurons = 2  # Two states for pendulum [theta, theta_dot]
num_features = 12  # Updated feature vector size for 2 states + control
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
# np.random.seed(12345)  # (optional) reproducibility of initial weights
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i}: {w}")

# --- EKF --- (Tuned parameters for pendulum)
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=1e-4, R_init=5e-3, P_init=1.0, eta=0.5
)
x_hat_ekf = np.zeros((n_steps, 2))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=1e-4, R_init=5e-3, P_init=1.0, eta=0.9,
    alpha=1e-2, beta=2.0  # UKF-specific parameters for pendulum
)
x_hat_ukf = np.zeros((n_steps, 2))
x_hat_ukf[0] = x_true[0]

# --- PF ---
n_particles = 300  # Particles for 2-state pendulum system

# State-specific noise parameters: [theta, theta_dot]
Q_std_per_state = [0.04, 0.08]  # Process noise: theta (smaller), theta_dot (larger)
R_std_per_state = [0.03, 0.06]  # Measurement noise: theta (smaller), theta_dot (larger)

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state, 
    ess_threshold=n_particles / 2  # ESS < N/2
)

# Display PF parameters for verification
pf_params = pf_trainer.get_parameters_info()
print("\nParticle Filter Parameters per State:")
for state, params in pf_params.items():
    print(f"  {state}: Q_std={params['Q_std']:.3f}, R_std={params['R_std']:.3f}, R_var={params['R_var']:.6f}")

# Force identical particle initialization if desired:
def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
    for i in range(pf_trainer_instance.num_neurons):
        pf_trainer_instance.particles[i] = np.tile(
            common_weights_list[i], (pf_trainer_instance.n_particles, 1)
        )
        pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

x_hat_pf = np.zeros((n_steps, 2))
x_hat_pf[0] = x_true[0]

print("\nStarting pendulum simulation...")
for k in range(n_steps - 1):
    # ---- 1) Generate control input and evolve true system -> k+1 ----
    t_current = k * dt
    u_current = generate_realistic_trajectory(t_current, trajectory_type)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, friction_variation, sensor_bias)

    # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ekf[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)

    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_state_for_z_ekf[0] = x_hat_ekf[k][0]  # series-parallel uses EKF's own estimate at k
    x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)  # theta
    x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)  # theta_dot

    # ---- 3) UKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ukf[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)

    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    x_state_for_z_ukf[0] = x_hat_ukf[k][0]  # series-parallel uses UKF's own estimate at k
    x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)  # theta
    x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)  # theta_dot

    # ---- 4) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_pf[k], x_hat_previous=x_hat_pf[k], u_input=u_current)

    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_state_for_z_pf[0] = x_hat_pf[k][0]  # series-parallel uses PF's own estimate at k
    x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], u_current)   # theta
    x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], u_current)   # theta_dot

    # if k % (n_steps // 10) == 0:
    #     print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [ 0.08482049  0.01578242  0.17398036 -0.28948468 -0.10254458 -0.19253029
  0.10869489 -0.09474529 -0.00327814  0.2417587   0.27853606 -0.12164859
  0.2089518  -0.23220261  0.18635296 -0.10265695  0.19580815 -0.08946135
 -0.02735337]
  Neuron 1: [ 0.13818381 -0.2098459  -0.03943097  0.27471141 -0.01515625  0.28915929
 -0.19671944 -0.12634501 -0.08285162 -0.03753638  0.00978873  0.16208158
  0.29864183 -0.23108392 -0.14093747 -0.02038229 -0.21365014 -0.21092492
  0.2221587 ]
  Neuron 2: [ 0.13808771 -0.25446662  0.13403806  0.06423545  0.26348678 -0.15121895
 -0.06431882  0.20071963 -0.00629371 -0.29033166 -0.22514769 -0.13883408
  0.23141317  0.00575692 -0.27367488 -0.21512602 -0.06595236 -0.17039152
  0.10954603]

Particle Filter Parameters per State:
  x: Q_std=0.025, R_std=0.015, R_var=0.000225
  y: Q_std=0.030, R_std=0.020, R_var=0.000400
  theta: Q_std=0.035, R_std=0.025, R_var=0.000625

Starting Lorenz chaotic system simulation...


Common Initial Weights:
  Neuron 0: [ 0.08482049  0.01578242  0.17398036 -0.28948468 -0.10254458 -0.19253029
  0.10869489 -0.09474529 -0.00327814  0.2417587   0.27853606 -0.12164859
  0.2089518  -0.23220261  0.18635296 -0.10265695  0.19580815 -0.08946135
 -0.02735337]
  Neuron 1: [ 0.13818381 -0.2098459  -0.03943097  0.27471141 -0.01515625  0.28915929
 -0.19671944 -0.12634501 -0.08285162 -0.03753638  0.00978873  0.16208158
  0.29864183 -0.23108392 -0.14093747 -0.02038229 -0.21365014 -0.21092492
  0.2221587 ]
  Neuron 2: [ 0.13808771 -0.25446662  0.13403806  0.06423545  0.26348678 -0.15121895
 -0.06431882  0.20071963 -0.00629371 -0.29033166 -0.22514769 -0.13883408
  0.23141317  0.00575692 -0.27367488 -0.21512602 -0.06595236 -0.17039152
  0.10954603]

Particle Filter Parameters per State:
  x: Q_std=0.025, R_std=0.015, R_var=0.000225
  y: Q_std=0.030, R_std=0.020, R_var=0.000400
  theta: Q_std=0.035, R_std=0.025, R_var=0.000625

Starting Lorenz chaotic system simulation...


/var/folders/kx/21rsqfy11f36xy7r66gt8fpc0000gn/T/ipykernel_4385/441949138.py:7: RuntimeWarning:

overflow encountered in exp



Common Initial Weights:
  Neuron 0: [ 0.08482049  0.01578242  0.17398036 -0.28948468 -0.10254458 -0.19253029
  0.10869489 -0.09474529 -0.00327814  0.2417587   0.27853606 -0.12164859
  0.2089518  -0.23220261  0.18635296 -0.10265695  0.19580815 -0.08946135
 -0.02735337]
  Neuron 1: [ 0.13818381 -0.2098459  -0.03943097  0.27471141 -0.01515625  0.28915929
 -0.19671944 -0.12634501 -0.08285162 -0.03753638  0.00978873  0.16208158
  0.29864183 -0.23108392 -0.14093747 -0.02038229 -0.21365014 -0.21092492
  0.2221587 ]
  Neuron 2: [ 0.13808771 -0.25446662  0.13403806  0.06423545  0.26348678 -0.15121895
 -0.06431882  0.20071963 -0.00629371 -0.29033166 -0.22514769 -0.13883408
  0.23141317  0.00575692 -0.27367488 -0.21512602 -0.06595236 -0.17039152
  0.10954603]

Particle Filter Parameters per State:
  x: Q_std=0.025, R_std=0.015, R_var=0.000225
  y: Q_std=0.030, R_std=0.020, R_var=0.000400
  theta: Q_std=0.035, R_std=0.025, R_var=0.000625

Starting Lorenz chaotic system simulation...


/var/folders/kx/21rsqfy11f36xy7r66gt8fpc0000gn/T/ipykernel_4385/441949138.py:7: RuntimeWarning:

overflow encountered in exp



Simulation finished.


In [17]:
# ============================================================
    # 6) Results & plots for Nonlinear Pendulum
# ============================================================
mse_theta_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)     # angular position
mse_thetadot_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)  # angular velocity

mse_theta_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)     # angular position
mse_thetadot_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)  # angular velocity

mse_theta_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)       # angular position
mse_thetadot_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)    # angular velocity

print(f"\nFinal EKF-RHONN Weights:")
for i in range(2):
    state_names = ['theta', 'theta_dot']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(2):
    state_names = ['theta', 'theta_dot']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(2):
    state_names = ['theta', 'theta_dot']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Nonlinear Pendulum ---")
print(f"EKF MSE theta:      {mse_theta_ekf:.6f}")
print(f"EKF MSE theta_dot:  {mse_thetadot_ekf:.6f}")
print(f"UKF MSE theta:      {mse_theta_ukf:.6f}")
print(f"UKF MSE theta_dot:  {mse_thetadot_ukf:.6f}")
print(f"PF  MSE theta:      {mse_theta_pf:.6f}")
print(f"PF  MSE theta_dot:  {mse_thetadot_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'theta', 'desc': 'Angular Position', 'y_label': 'θ (rad)',
     'chi': 'χθ (True θ)', 'x': 'θ (Est.)'},
    {'idx': 1, 'var': 'theta_dot', 'desc': 'Angular Velocity', 'y_label': 'θ̇ (rad/s)',
     'chi': 'χθ̇ (True θ̇)', 'x': 'θ̇ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines',
                        name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))

    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(
        title=f'Pendulum RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for both states
error_theta_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_thetadot_ekf = x_true[:, 1] - x_hat_ekf[:, 1]

error_theta_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_thetadot_ukf = x_true[:, 1] - x_hat_ukf[:, 1]

error_theta_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_thetadot_pf = x_true[:, 1] - x_hat_pf[:, 1]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines',
                        name=f'EKF Error θ (MSE={mse_theta_ekf:.6f})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ukf, mode='lines',
                        name=f'UKF Error θ (MSE={mse_theta_ukf:.6f})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines',
                        name=f'PF Error θ (MSE={mse_theta_pf:.6f})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_thetadot_ekf, mode='lines',
                        name=f'EKF Error θ̇ (MSE={mse_thetadot_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_thetadot_ukf, mode='lines',
                        name=f'UKF Error θ̇ (MSE={mse_thetadot_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_thetadot_pf, mode='lines',
                        name=f'PF Error θ̇ (MSE={mse_thetadot_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.update_layout(
    title='Pendulum RHONN Identification Errors (EKF vs UKF vs PF)',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# Phase plot (theta vs theta_dot for pendulum)
fig_phase = go.Figure()
fig_phase.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines',
                              name='True Pendulum Phase Plot',
                              line=dict(color='black', width=3)))
fig_phase.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], mode='lines',
                              name='EKF Estimation',
                              line=dict(color='blue', width=2, dash='dash')))
fig_phase.add_trace(go.Scatter(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], mode='lines',
                              name='UKF Estimation',
                              line=dict(color='green', width=2, dash='dashdot')))
fig_phase.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], mode='lines',
                              name='PF Estimation',
                              line=dict(color='red', width=2, dash='dot')))
# Add start and end markers
fig_phase.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers',
                              name='Start', marker=dict(color='green', size=10, symbol='star')))
fig_phase.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]], mode='markers',
                              name='End', marker=dict(color='red', size=10, symbol='square')))
fig_phase.update_layout(
    title='Pendulum Phase Plot - State Space Comparison (EKF vs UKF vs PF)',
    xaxis_title='θ (rad)',
    yaxis_title='θ̇ (rad/s)',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True
)
fig_phase.show()

# Determine which filter has the lowest total MSE (sum of theta, theta_dot)
mse_total_ekf = mse_theta_ekf + mse_thetadot_ekf
mse_total_ukf = mse_theta_ukf + mse_thetadot_ukf
mse_total_pf = mse_theta_pf + mse_thetadot_pf

mse_totals = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
best_filter = min(mse_totals, key=mse_totals.get)

print(f"\nBest overall performance: {best_filter} (lowest total MSE: {mse_totals[best_filter]:.6f})")


Final EKF-RHONN Weights:
  Neuron 1 (x): [-2.8015582   1.87350159 -0.20778889  1.01669333 -3.17233639  0.98444276
  2.88293985  1.99456635 -1.27382321  2.2345489  -0.91136771 -0.48829241
  0.21605472 -0.71171701 -0.20026476  1.4728702  -0.17633135 -0.08946135
 -0.8005888 ]
  Neuron 2 (y): [-1.94347064  2.95812498 -1.79160731  3.04689394 -3.22968777  2.38953542
  1.89325819  3.82188399 -3.13269771  4.72376681 -1.4918328  -1.15135927
 -0.4496866  -1.54670311 -1.50249199  2.43326881 -0.0299842  -0.21092492
 -2.50095036]
  Neuron 3 (z): [ 2.72230243e-03  1.35660513e+00 -3.29271948e+00 -4.75037517e-01
 -4.35881707e-01 -8.73819241e-01 -2.78059710e+00  3.96037903e+00
 -2.58330734e+00  2.31357435e+00 -1.03516394e+00  8.06756442e-01
  3.04981357e-01  2.43677921e-01  1.78484320e-01 -1.73035277e+00
 -9.23282852e-01 -1.70391516e-01  1.01386444e+00]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 2.79433145 -0.21496095  1.04259409 -0.29221406 -0.6641355   0.08735733
  0.00808664  0.47262604  0.6653257


Final EKF-RHONN Weights:
  Neuron 1 (x): [-2.8015582   1.87350159 -0.20778889  1.01669333 -3.17233639  0.98444276
  2.88293985  1.99456635 -1.27382321  2.2345489  -0.91136771 -0.48829241
  0.21605472 -0.71171701 -0.20026476  1.4728702  -0.17633135 -0.08946135
 -0.8005888 ]
  Neuron 2 (y): [-1.94347064  2.95812498 -1.79160731  3.04689394 -3.22968777  2.38953542
  1.89325819  3.82188399 -3.13269771  4.72376681 -1.4918328  -1.15135927
 -0.4496866  -1.54670311 -1.50249199  2.43326881 -0.0299842  -0.21092492
 -2.50095036]
  Neuron 3 (z): [ 2.72230243e-03  1.35660513e+00 -3.29271948e+00 -4.75037517e-01
 -4.35881707e-01 -8.73819241e-01 -2.78059710e+00  3.96037903e+00
 -2.58330734e+00  2.31357435e+00 -1.03516394e+00  8.06756442e-01
  3.04981357e-01  2.43677921e-01  1.78484320e-01 -1.73035277e+00
 -9.23282852e-01 -1.70391516e-01  1.01386444e+00]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 2.79433145 -0.21496095  1.04259409 -0.29221406 -0.6641355   0.08735733
  0.00808664  0.47262604  0.6653257


Final EKF-RHONN Weights:
  Neuron 1 (x): [-2.8015582   1.87350159 -0.20778889  1.01669333 -3.17233639  0.98444276
  2.88293985  1.99456635 -1.27382321  2.2345489  -0.91136771 -0.48829241
  0.21605472 -0.71171701 -0.20026476  1.4728702  -0.17633135 -0.08946135
 -0.8005888 ]
  Neuron 2 (y): [-1.94347064  2.95812498 -1.79160731  3.04689394 -3.22968777  2.38953542
  1.89325819  3.82188399 -3.13269771  4.72376681 -1.4918328  -1.15135927
 -0.4496866  -1.54670311 -1.50249199  2.43326881 -0.0299842  -0.21092492
 -2.50095036]
  Neuron 3 (z): [ 2.72230243e-03  1.35660513e+00 -3.29271948e+00 -4.75037517e-01
 -4.35881707e-01 -8.73819241e-01 -2.78059710e+00  3.96037903e+00
 -2.58330734e+00  2.31357435e+00 -1.03516394e+00  8.06756442e-01
  3.04981357e-01  2.43677921e-01  1.78484320e-01 -1.73035277e+00
 -9.23282852e-01 -1.70391516e-01  1.01386444e+00]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 2.79433145 -0.21496095  1.04259409 -0.29221406 -0.6641355   0.08735733
  0.00808664  0.47262604  0.6653257


Final EKF-RHONN Weights:
  Neuron 1 (x): [-2.8015582   1.87350159 -0.20778889  1.01669333 -3.17233639  0.98444276
  2.88293985  1.99456635 -1.27382321  2.2345489  -0.91136771 -0.48829241
  0.21605472 -0.71171701 -0.20026476  1.4728702  -0.17633135 -0.08946135
 -0.8005888 ]
  Neuron 2 (y): [-1.94347064  2.95812498 -1.79160731  3.04689394 -3.22968777  2.38953542
  1.89325819  3.82188399 -3.13269771  4.72376681 -1.4918328  -1.15135927
 -0.4496866  -1.54670311 -1.50249199  2.43326881 -0.0299842  -0.21092492
 -2.50095036]
  Neuron 3 (z): [ 2.72230243e-03  1.35660513e+00 -3.29271948e+00 -4.75037517e-01
 -4.35881707e-01 -8.73819241e-01 -2.78059710e+00  3.96037903e+00
 -2.58330734e+00  2.31357435e+00 -1.03516394e+00  8.06756442e-01
  3.04981357e-01  2.43677921e-01  1.78484320e-01 -1.73035277e+00
 -9.23282852e-01 -1.70391516e-01  1.01386444e+00]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 2.79433145 -0.21496095  1.04259409 -0.29221406 -0.6641355   0.08735733
  0.00808664  0.47262604  0.6653257


Final EKF-RHONN Weights:
  Neuron 1 (x): [-2.8015582   1.87350159 -0.20778889  1.01669333 -3.17233639  0.98444276
  2.88293985  1.99456635 -1.27382321  2.2345489  -0.91136771 -0.48829241
  0.21605472 -0.71171701 -0.20026476  1.4728702  -0.17633135 -0.08946135
 -0.8005888 ]
  Neuron 2 (y): [-1.94347064  2.95812498 -1.79160731  3.04689394 -3.22968777  2.38953542
  1.89325819  3.82188399 -3.13269771  4.72376681 -1.4918328  -1.15135927
 -0.4496866  -1.54670311 -1.50249199  2.43326881 -0.0299842  -0.21092492
 -2.50095036]
  Neuron 3 (z): [ 2.72230243e-03  1.35660513e+00 -3.29271948e+00 -4.75037517e-01
 -4.35881707e-01 -8.73819241e-01 -2.78059710e+00  3.96037903e+00
 -2.58330734e+00  2.31357435e+00 -1.03516394e+00  8.06756442e-01
  3.04981357e-01  2.43677921e-01  1.78484320e-01 -1.73035277e+00
 -9.23282852e-01 -1.70391516e-01  1.01386444e+00]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 2.79433145 -0.21496095  1.04259409 -0.29221406 -0.6641355   0.08735733
  0.00808664  0.47262604  0.6653257


Final EKF-RHONN Weights:
  Neuron 1 (x): [-2.8015582   1.87350159 -0.20778889  1.01669333 -3.17233639  0.98444276
  2.88293985  1.99456635 -1.27382321  2.2345489  -0.91136771 -0.48829241
  0.21605472 -0.71171701 -0.20026476  1.4728702  -0.17633135 -0.08946135
 -0.8005888 ]
  Neuron 2 (y): [-1.94347064  2.95812498 -1.79160731  3.04689394 -3.22968777  2.38953542
  1.89325819  3.82188399 -3.13269771  4.72376681 -1.4918328  -1.15135927
 -0.4496866  -1.54670311 -1.50249199  2.43326881 -0.0299842  -0.21092492
 -2.50095036]
  Neuron 3 (z): [ 2.72230243e-03  1.35660513e+00 -3.29271948e+00 -4.75037517e-01
 -4.35881707e-01 -8.73819241e-01 -2.78059710e+00  3.96037903e+00
 -2.58330734e+00  2.31357435e+00 -1.03516394e+00  8.06756442e-01
  3.04981357e-01  2.43677921e-01  1.78484320e-01 -1.73035277e+00
 -9.23282852e-01 -1.70391516e-01  1.01386444e+00]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 2.79433145 -0.21496095  1.04259409 -0.29221406 -0.6641355   0.08735733
  0.00808664  0.47262604  0.6653257


Best overall performance: UKF (lowest total MSE: 7.991927)
PF superiority factor vs EKF: 3216367719564639934468752132716736715150260001323026367164453763127449662590649238253986908844531619988866145845718137140066047497015572257731083434020073755622097332090569899141758976.00x
PF superiority factor vs UKF: 0.30x
